<a href="https://colab.research.google.com/github/kp-algo/encoder-decoder/blob/main/encoder_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tokenizers
!pip install datasets
#packages
import numpy as np
import urllib.request, zipfile
import torch
import torch.nn as nn
import torch.nn.functional as f
from tokenizers import ByteLevelBPETokenizer, pre_tokenizers
from tokenizers.pre_tokenizers import Whitespace
import itertools
from datasets import load_dataset

#Load Dataset

In [2]:
n_train_stories, n_val_stories = 50_000, 5_000     # ~tens of MB of text, plenty for a 3M-param model

def grab(split, n):
    ds = load_dataset("roneneldan/TinyStories", split=split, streaming=True)
    return [ex["text"] for ex in itertools.islice(ds, n)]

train_stories = grab("train", n_train_stories)
val_stories   = grab("validation", n_val_stories)

print(len(train_stories), len(val_stories))
print(train_stories[0][:300])

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

50000 5000
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and 


#Tokenisation

In [3]:
EOS = "<eos>"
train_text = EOS.join(train_stories)
val_text   = EOS.join(val_stories)

In [4]:
tokenizer = ByteLevelBPETokenizer()

tokenizer.train_from_iterator(
    train_stories, vocab_size=3000, min_frequency=2,
    special_tokens=["<bos>", "<mask>", EOS],
)

encode = lambda s: tokenizer.encode(s).ids
decode = lambda ids: tokenizer.decode(ids)

vocab_size = tokenizer.get_vocab_size()
bos_id, mask_id, eos_id = (tokenizer.token_to_id(t) for t in ["<bos>", "<mask>", EOS])

train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data   = torch.tensor(encode(val_text),   dtype=torch.long)
print(vocab_size, len(train_data), len(val_data))

3000 11745267 1061769


#Hyperparameters

In [5]:
#hyperparams
torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 192
block_size = 128
dropout = 0.2
head_size = 48
n_head = 4
n_layers = 6
vocab_size = tokenizer.get_vocab_size()
batch_size = 32
learning_rate = 3e-4
src_len = 128            # characters the encoder reads
tgt_len = 128
max_iters = 35000
eval_interval = 500
eval_iters = 100

# Data Split



*   BART: requries two different inputs stream(one for encoder and one for decoder)
*   GPT: requires a single input stream(for decoder alone)



In [6]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - src_len - tgt_len - 1, (batch_size,))
    src = torch.stack([d[i : i + src_len] for i in ix])
    tgt = torch.stack([d[i + src_len : i + src_len + tgt_len] for i in ix])
    bos = torch.full((batch_size, 1), bos_id, dtype=torch.long)
    dec_in = torch.cat([bos, tgt[:, :-1]], dim=1)
    return src.to(device), dec_in.to(device), tgt.to(device)

In [7]:
def get_batch_gpt(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

In [8]:
def mask_batch(x, p=0.15):
    m = torch.rand(x.shape, device=x.device) < p
    inp = x.clone();  inp[m] = mask_id # hide 15% of characters
    tgt = x.clone();  tgt[~m] = -100  # loss only on the hidden ones
    return inp, tgt

#Architecture


*   Head: single attention head; fundamental unit of the architecture
*   MultiHead: stacks the multiple heads
*   FeedForward: simple linear neural network
*   Encoder Block: encoder; no masking
*   Decoder Block: decoder; masking
*   GPT: encoder-decoder; final model







In [9]:
class Head(nn.Module):
    def __init__(self, head_size, causal):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        self.causal = causal

    def forward(self, x, context=None):
        B,T,C = x.shape

        q = self.query(x)

        cxt = x if context is None else context

        k = self.key(cxt)
        v = self.value(cxt)

        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        if self.causal:
            wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        wei = f.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out


In [10]:
class MultiHead(nn.Module):
    def __init__(self, head_size, num_heads, causal):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, causal) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, context=None):
        out = torch.cat([head(x, context) for head in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)

        return out

In [11]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd * 4),
            nn.ReLU(),
            nn.Linear(n_embd * 4, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        out = self.net(x)
        return out

In [12]:
class EncoderBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd//n_head
        self.sa_head = MultiHead(head_size, n_head, causal=False)
        self.fwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa_head(self.ln1(x))
        x = x + self.fwd(self.ln2(x))
        return x

In [13]:
class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd//n_head
        self.sa_head = MultiHead(head_size, n_head, causal = True)
        self.ca_head = MultiHead(head_size, n_head, causal = False)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)
        self.fwd = FeedForward(n_embd)

    def forward(self, x, enc):
        x = x + self.sa_head(self.ln1(x))
        x = x + self.ca_head(self.ln2(x), enc)
        x = x + self.fwd(self.ln3(x))
        return x

#BART:


---


*   an autoregressive model with both **encoder** and **decoder** block connected with cross attention.
*   requires two separate inputs; one for encoder and other one for decoder.



In [14]:
class BART(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embd = nn.Embedding(vocab_size, n_embd)
        self.pos_embd = nn.Embedding(block_size, n_embd)

        self.enc_blocks = nn.Sequential(*[EncoderBlock(n_embd, n_head) for _ in range(n_layers)])
        self.dec_blocks = nn.ModuleList([DecoderBlock(n_embd, n_head) for _ in range(n_layers)])

        self.ln_enc = nn.LayerNorm(n_embd)
        self.lnf = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def embed(self, ids):
        T = ids.shape[1]
        return self.token_embd(ids) + self.pos_embd(torch.arange(T, device=ids.device))

    def encode(self, src):
        return self.ln_enc(self.enc_blocks(self.embed(src)))

    def decode(self, dec_in, enc):
        x = self.embed(dec_in)
        for dec in self.dec_blocks:
            x = dec(x, enc)
        return self.lm_head(self.lnf(x))

    def forward(self, src, dec_in, targets=None):
        logits = self.decode(dec_in, self.encode(src))
        loss = None if targets is None else f.cross_entropy(
            logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss

    #claude generated
    @torch.no_grad()
    def generate(self, src, bos_id, max_new_tokens=64, temperature=0.8, top_k=40):
        self.eval()                                    # turns dropout off
        enc = self.encode(src)                         # (B, S, n_embd), computed once
        out = torch.full((src.shape[0], 1), bos_id, dtype=torch.long, device=src.device)

        for _ in range(max_new_tokens):
            dec_in = out[:, -block_size:]              # crop to the position table size
            logits = self.decode(dec_in, enc)[:, -1, :] / temperature     # last position only

            if top_k is not None:                      # keep only the k most likely tokens
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = f.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            out = torch.cat([out, next_tok], dim=1)

        return out[:, 1:]                              # drop the start token

In [15]:
m = BART().to(device)                         # your model class name
print(sum(p.numel() for p in m.parameters()) / 1e6, 'M parameters')

src, dec_in, tgt = get_batch('train')
logits, loss = m(src, dec_in, tgt)
print(logits.shape, loss.item())

7.399992 M parameters
torch.Size([32, 128, 3000]) 8.206897735595703


#GPT:


---
*   an autoregressive model with n_layers of **decoder** blocks  
*   requires a single input stream for decoder


In [24]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embd = nn.Embedding(vocab_size, n_embd)
        self.pos_embd = nn.Embedding(block_size, n_embd)

        head_size = n_embd // n_head
        self.sa  = nn.ModuleList([MultiHead(head_size, n_head, causal=True) for _ in range(n_layers)])
        self.fwd = nn.ModuleList([FeedForward(n_embd) for _ in range(n_layers)])
        self.ln1 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layers)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layers)])

        self.lnf = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embd(idx) + self.pos_embd(torch.arange(T, device=idx.device))

        for sa, fwd, ln1, ln2 in zip(self.sa, self.fwd, self.ln1, self.ln2):
            x = x + sa(ln1(x))
            x = x + fwd(ln2(x))

        logits = self.lm_head(self.lnf(x))          # (B, T, vocab_size)
        loss = None if targets is None else f.cross_entropy(
            logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = f.softmax(logits[:, -1, :] / temperature, dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
        return idx

In [25]:
gpt = GPT().to(device)
print(sum(p.numel() for p in gpt.parameters()) / 1e6, 'M parameters')

src, dec_in, tgt = get_batch('train')
logits, loss = gpt(src, tgt)
print(logits.shape, loss.item())

3.845688 M parameters
torch.Size([32, 128, 3000]) 8.152883529663086


#BERT


---
*   a bidirectional model with n_layers of encoder block
*   requires a single stream masked input



In [26]:
class BERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embd = nn.Embedding(vocab_total, n_embd)
        self.pos_embd = nn.Embedding(block_size, n_embd)

        head_size = n_embd // n_head
        self.sa  = nn.ModuleList([MultiHead(head_size, n_head, causal=False) for _ in range(n_layers)])
        self.fwd = nn.ModuleList([FeedForward(n_embd) for _ in range(n_layers)])
        self.ln1 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layers)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layers)])

        self.lnf = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_total)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embd(idx) + self.pos_embd(torch.arange(T, device=idx.device))

        for sa, fwd, ln1, ln2 in zip(self.sa, self.fwd, self.ln1, self.ln2):
            x = x + sa(ln1(x))
            x = x + fwd(ln2(x))

        logits = self.lm_head(self.lnf(x))          # (B, T, vocab_total)
        loss = None if targets is None else f.cross_entropy(
            logits.view(-1, logits.shape[-1]), targets.view(-1), ignore_index=-100)
        return logits, loss

#Output:


---



*   BART(with continuation)
*   GPT
*   BERT




In [19]:
@torch.no_grad()
def estimate_loss():
    m.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            s, d_in, t = get_batch(split)
            _, loss = m(s, d_in, t)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    m.train()
    return out

optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        L = estimate_loss()
        print(f"step {it}: train {L['train']:.3f}, val {L['val']:.3f}")

    s, d_in, t = get_batch('train')
    _, loss = m(s, d_in, t)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train 8.216, val 8.214
step 500: train 4.079, val 4.125
step 1000: train 3.599, val 3.640
step 1500: train 3.322, val 3.381
step 2000: train 3.170, val 3.234
step 2500: train 3.042, val 3.099
step 3000: train 2.939, val 3.020
step 3500: train 2.872, val 2.934
step 4000: train 2.807, val 2.870
step 4500: train 2.757, val 2.816
step 5000: train 2.714, val 2.764
step 5500: train 2.636, val 2.702
step 6000: train 2.604, val 2.662
step 6500: train 2.567, val 2.622
step 7000: train 2.534, val 2.610
step 7500: train 2.486, val 2.569
step 8000: train 2.463, val 2.537
step 8500: train 2.440, val 2.514
step 9000: train 2.409, val 2.474
step 9500: train 2.369, val 2.449
step 10000: train 2.369, val 2.445
step 10500: train 2.341, val 2.422
step 11000: train 2.313, val 2.400
step 11500: train 2.291, val 2.386
step 12000: train 2.272, val 2.364
step 12500: train 2.271, val 2.341
step 13000: train 2.259, val 2.346
step 13500: train 2.224, val 2.299
step 14000: train 2.211, val 2.307
step 1450

In [20]:
src, _, tgt = get_batch('val')
gen = m.generate(src[:1], bos_id, max_new_tokens=200)

print('SOURCE:\n', decode(src[0].tolist()))
print('\nACTUAL:\n', decode(tgt[0].tolist()))
print('\nMODEL:\n', decode(gen[0].tolist()))

SOURCE:
 �© was yelling at another young princess. 

The young princess felt so sad. She knew she had to do something. She walked closer and confronted her fiancÃ©. He said that he had decided to marry the other princess instead. 

The young princess was so upset that she ran away. She ran and ran until she got far away from the castle. She was so sad that she never wanted to get married again. She never saw her fiancÃ© or the other young princess ever again. 

The young princess lived her life alone

ACTUAL:
 , never finding true love.Timmy was a very obedient boy. He was always doing what his parents asked. One day, his parents asked him to go to the science lab. Timmy was scared but went anyway.

When Timmy got to the science lab he saw a big green laser. "What is that?" he asked his parents. His parents explained that it was a laser and the scientists were going to use it to test things.

Timmy was still scared but his parents reassured him that the laser was safe. They asked

MODE

#GPT

In [21]:
@torch.no_grad()
def estimate_loss_gpt(model):
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch_gpt(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out
optimizer = torch.optim.AdamW(gpt.parameters(), lr=learning_rate)

for it in range(max_iters):
    x, y = get_batch_gpt('train')
    _, loss = gpt(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
    optimizer.step()

    if it % eval_interval == 0 or it == max_iters - 1:
        L = estimate_loss_gpt(gpt)
        print(f"step {it}: train {L['train']:.3f}, val {L['val']:.3f}")

step 0: train 7.965, val 7.971
step 500: train 4.035, val 4.070
step 1000: train 3.567, val 3.632
step 1500: train 3.309, val 3.331
step 2000: train 3.146, val 3.205
step 2500: train 3.043, val 3.083
step 3000: train 2.945, val 3.012
step 3500: train 2.878, val 2.940
step 4000: train 2.823, val 2.874
step 4500: train 2.770, val 2.820
step 5000: train 2.732, val 2.768
step 5500: train 2.689, val 2.750
step 6000: train 2.643, val 2.703
step 6500: train 2.610, val 2.664
step 7000: train 2.586, val 2.653
step 7500: train 2.546, val 2.609
step 8000: train 2.515, val 2.584
step 8500: train 2.497, val 2.578
step 9000: train 2.481, val 2.555
step 9500: train 2.457, val 2.532
step 10000: train 2.431, val 2.512
step 10500: train 2.420, val 2.498
step 11000: train 2.387, val 2.461
step 11500: train 2.394, val 2.459
step 12000: train 2.376, val 2.433
step 12500: train 2.345, val 2.430
step 13000: train 2.348, val 2.417
step 13500: train 2.342, val 2.413
step 14000: train 2.325, val 2.400
step 1450

In [22]:
src, _, tgt = get_batch('val')
gen = m.generate(src[:1], bos_id, max_new_tokens=200)

print('SOURCE:\n', decode(src[0].tolist()))
print('\nACTUAL:\n', decode(tgt[0].tolist()))
print('\nMODEL:\n', decode(gen[0].tolist()))

SOURCE:
  cut go away."

Sara smiles. She says, "Thank you, Ben. You are a good friend. You are kind." She hugs Ben and the bear. Ben hugs Sara and the car. They are happy again. They play with the bear and the car until it is time to go home. They say, "Bye, box. Thank you for the unusual things. We had fun." They take the bear and the car with them. They are friends.Sara and Tom were friends. They liked to play in the park. One day, they saw a big bunch of grapes on a

ACTUAL:
  table. They were hungry and wanted to eat some.

"Look, grapes!" Sara said. "Let's go and get some."

"But they are not ours," Tom said. "Maybe someone else will come back for them."

Sara did not listen. She ran to the table and grabbed a grape. She put it in her mouth. It was sweet and juicy. She was glad.

"Come on, Tom, they are yummy!" Sara said. "No one is here. We can have some more."

Tom was scared. He did not want to

MODEL:
  wire on the ground. It looked like a wheel and a shiny wire.

"Let's pick

#BERT

In [23]:
@torch.no_grad()
def estimate_bert(model):
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses, accs = torch.zeros(eval_iters), torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, _ = get_batch_gpt(split)
            inp, tgt = mask_batch(x)
            logits, loss = model(inp, tgt)
            sel = tgt != -100
            accs[k] = (logits.argmax(-1)[sel] == tgt[sel]).float().mean()
            losses[k] = loss.item()
        out[split] = (losses.mean().item(), accs.mean().item())
    model.train()
    return out


bert = BERT(n_embd, n_head, n_layers).to(device)
optimizer = torch.optim.AdamW(bert.parameters(), lr=learning_rate)   # same cell as the model

for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        L = estimate_bert(bert)
        print(f"step {it}: train {L['train'][0]:.3f} (acc {L['train'][1]:.3f}), "
              f"val {L['val'][0]:.3f} (acc {L['val'][1]:.3f})")

    x, _ = get_batch_gpt('train')
    inp, tgt = mask_batch(x)
    _, loss = bert(inp, tgt)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(bert.parameters(), 1.0)
    optimizer.step()

step 0: train 8.159 (acc 0.000), val 8.160 (acc 0.001)
step 500: train 6.009 (acc 0.069), val 6.034 (acc 0.066)
step 1000: train 5.898 (acc 0.071), val 5.983 (acc 0.063)
step 1500: train 5.880 (acc 0.070), val 5.938 (acc 0.065)
step 2000: train 5.864 (acc 0.069), val 5.905 (acc 0.065)
step 2500: train 5.835 (acc 0.069), val 5.876 (acc 0.067)
step 3000: train 5.830 (acc 0.068), val 5.893 (acc 0.064)
step 3500: train 5.772 (acc 0.077), val 5.824 (acc 0.070)
step 4000: train 5.328 (acc 0.123), val 5.397 (acc 0.115)
step 4500: train 4.869 (acc 0.174), val 4.911 (acc 0.170)
step 5000: train 4.575 (acc 0.206), val 4.623 (acc 0.201)
step 5500: train 4.179 (acc 0.256), val 4.257 (acc 0.251)
step 6000: train 3.908 (acc 0.293), val 3.970 (acc 0.288)
step 6500: train 3.698 (acc 0.319), val 3.772 (acc 0.314)
step 7000: train 3.539 (acc 0.340), val 3.599 (acc 0.334)
step 7500: train 3.392 (acc 0.358), val 3.447 (acc 0.352)
step 8000: train 3.297 (acc 0.369), val 3.349 (acc 0.363)
step 8500: train 3